# AI Laboratory: Logical Planning

**Learning Objectives:**
1. Describe a simple planning problem using states, actions, preconditions, and effects
2. Determine whether an action is logically applicable in a given state
3. Explain how logical reasoning and search work together in planning
4. Use an LLM to generate a simple planning program
5. Test and validate LLM-generated code
6. Identify limitations of relying on an LLM to generate an intelligent agent
7. (Optional) Use Prolog as a logical verifier

**Central Idea:** Logic + Search = Planning

**Key Principle:** Logic determines what is possible; search determines what to try.


## Task 0: Understand the Planning Problem

Before using an LLM, we write down the planning problem formally.

### Problem Specification: Warehouse Robot Package Delivery

**A planning problem is described by:** $(I, A, G)$

where:
- **I** = Initial state
- **A** = Set of available actions
- **G** = Goal state

### Our Problem:

**Initial State I:**
```
I = {At(Robot, A), At(Package, A)}
```

**Goal G:**
```
G = {At(Package, C)}
```

**Available Actions:**

1. **Move(from, to)**
   - Preconditions: At(Robot, from)
   - Effects: ¬At(Robot, from), At(Robot, to)
   - Valid moves: (A,B), (B,A), (B,C), (C,B)

2. **PickUp(package, location)**
   - Preconditions: At(Robot, location), At(Package, location)
   - Effects: ¬At(Package, location), Holding(Package)

3. **Drop(package, location)**
   - Preconditions: At(Robot, location), Holding(Package)
   - Effects: ¬Holding(Package), At(Package, location)

### Key Questions:

**(a) Is PickUp(Package, A) applicable in the initial state?**
```
Check: At(Robot, A) ∈ I? YES ✓
Check: At(Package, A) ∈ I? YES ✓
Answer: YES, the action is applicable
```

**(b) Is Drop(Package, C) applicable in the initial state?**
```
Check: At(Robot, C) ∈ I? NO ✗
Check: Holding(Package) ∈ I? NO ✗
Answer: NO, preconditions not satisfied
```

**Key Insight:** An action is NOT applicable just because it's in the list of available actions. We must check if ALL preconditions are satisfied by the current state.

## Task 1: Construct a Plan by Hand

Before writing any program, we manually construct a valid plan.

A plan is a sequence of actions $a_1, a_2, ..., a_n$ such that:
$$I \xrightarrow{a_1} S_1 \xrightarrow{a_2} S_2 \rightarrow ... \rightarrow S_n$$
and $S_n \models G$ (goal is satisfied in final state)

### Hand-Constructed Plan:

| Step | Action | Preconditions | State After Action |
|------|--------|---------------|--------------------|
| **S₀** (initial) | — | — | At(Robot, A), At(Package, A) |
| 1 | PickUp(Package, A) | At(Robot, A) ✓, At(Package, A) ✓ | At(Robot, A), Holding(Package) |
| 2 | Move(A, B) | At(Robot, A) ✓ | At(Robot, B), Holding(Package) |
| 3 | Move(B, C) | At(Robot, B) ✓ | At(Robot, C), Holding(Package) |
| **S₃** (final) | Drop(Package, C) | At(Robot, C) ✓, Holding(Package) ✓ | At(Robot, C), At(Package, C) |

**Goal Check:** At(Package, C) ∈ S₃? **YES ✓**

**Plan:** [PickUp(Package, A), Move(A, B), Move(B, C), Drop(Package, C)]

**Length:** 4 actions

## Task 2 & 3: LLM Prompt and Reference Implementation

### Prompt Sent to LLM:

```
Write a simple planning agent in Python.

Represent a state as a set of logical propositions.

Each action should contain:
  - a name
  - positive preconditions (must be true)
  - negative preconditions (must be false)
  - positive effects (added to state)
  - negative effects (removed from state)

An action is applicable if all preconditions are satisfied.
When applied:
  1. Remove negative effects from state
  2. Add positive effects to state

Use breadth-first search to find a sequence of actions.

The program should:
  - Detect when no plan exists
  - Print the action sequence
  - Print the states after each action

Problem:
  Initial: At(Robot,A), At(Package,A)
  Goal: At(Package,C)
  Actions: Move, PickUp, Drop (defined as above)
```

## Implement Reference Planner

We implement a trusted version to validate LLM-generated code.

In [ ]:
from collections import deque
from copy import deepcopy

class Action:
    """Represents a STRIPS-style action with preconditions and effects."""
    
    def __init__(self, name, pos_pre, neg_pre, pos_eff, neg_eff):
        self.name = name
        self.pos_pre = pos_pre  # Positive preconditions (must be in state)
        self.neg_pre = neg_pre  # Negative preconditions (must NOT be in state)
        self.pos_eff = pos_eff  # Positive effects (add to state)
        self.neg_eff = neg_eff  # Negative effects (remove from state)
    
    def is_applicable(self, state):
        """Check if action can be applied in this state."""
        # All positive preconditions must be in state
        for pre in self.pos_pre:
            if pre not in state:
                return False
        # All negative preconditions must NOT be in state
        for pre in self.neg_pre:
            if pre in state:
                return False
        return True
    
    def apply(self, state):
        """Apply action to state, return new state."""
        new_state = state.copy()
        # Remove negative effects
        for eff in self.neg_eff:
            new_state.discard(eff)
        # Add positive effects
        for eff in self.pos_eff:
            new_state.add(eff)
        return new_state
    
    def __repr__(self):
        return self.name

# Define the warehouse problem
initial_state = {
    "At(Robot,A)",
    "At(Package,A)"
}

goal = {
    "At(Package,C)"
}

# Define actions
actions = [
    # Move actions
    Action("Move(A,B)", {"At(Robot,A)"}, set(), {"At(Robot,B)"}, {"At(Robot,A)"}),
    Action("Move(B,A)", {"At(Robot,B)"}, set(), {"At(Robot,A)"}, {"At(Robot,B)"}),
    Action("Move(B,C)", {"At(Robot,B)"}, set(), {"At(Robot,C)"}, {"At(Robot,B)"}),
    Action("Move(C,B)", {"At(Robot,C)"}, set(), {"At(Robot,B)"}, {"At(Robot,C)"}),
    
    # PickUp actions
    Action("PickUp(Package,A)",
           {"At(Robot,A)", "At(Package,A)"},
           set(),
           {"Holding(Package)"},
           {"At(Package,A)"}),
    Action("PickUp(Package,B)",
           {"At(Robot,B)", "At(Package,B)"},
           set(),
           {"Holding(Package)"},
           {"At(Package,B)"}),
    Action("PickUp(Package,C)",
           {"At(Robot,C)", "At(Package,C)"},
           set(),
           {"Holding(Package)"},
           {"At(Package,C)"}),
    
    # Drop actions
    Action("Drop(Package,A)",
           {"At(Robot,A)", "Holding(Package)"},
           set(),
           {"At(Package,A)"},
           {"Holding(Package)"}),
    Action("Drop(Package,B)",
           {"At(Robot,B)", "Holding(Package)"},
           set(),
           {"At(Package,B)"},
           {"Holding(Package)"}),
    Action("Drop(Package,C)",
           {"At(Robot,C)", "Holding(Package)"},
           set(),
           {"At(Package,C)"},
           {"Holding(Package)"}),
]

print("✓ Warehouse problem defined")
print(f"  Initial state: {initial_state}")
print(f"  Goal: {goal}")
print(f"  Actions: {len(actions)}")

In [ ]:
class SimplePlanner:
    """BFS-based STRIPS planner."""
    
    def __init__(self, initial_state, goal, actions):
        self.initial_state = initial_state
        self.goal = goal
        self.actions = actions
        self.states_explored = 0
    
    def goal_satisfied(self, state):
        """Check if goal is satisfied in this state."""
        return self.goal.issubset(state)
    
    def bfs_plan(self):
        """Find a plan using Breadth-First Search."""
        self.states_explored = 0
        
        queue = deque([(self.initial_state, [])])  # (state, plan)
        visited = {frozenset(self.initial_state)}
        
        while queue:
            current_state, plan = queue.popleft()
            self.states_explored += 1
            
            # Check if goal satisfied
            if self.goal_satisfied(current_state):
                return plan, current_state
            
            # Try all applicable actions
            for action in self.actions:
                if action.is_applicable(current_state):
                    new_state = action.apply(current_state)
                    state_frozen = frozenset(new_state)
                    
                    if state_frozen not in visited:
                        visited.add(state_frozen)
                        queue.append((new_state, plan + [action]))
        
        return None, None  # No plan found
    
    def print_plan(self, plan, final_state):
        """Print plan and states."""
        if plan is None:
            print("No plan found!")
            return
        
        print(f"\n✓ Plan found ({len(plan)} actions):\n")
        
        # Print initial state
        print(f"S₀: {self.initial_state}")
        
        # Apply each action and print result
        current_state = self.initial_state.copy()
        for i, action in enumerate(plan, 1):
            print(f"\nAction: {action}")
            print(f"  Preconditions: pos={action.pos_pre}, neg={action.neg_pre}")
            print(f"  Effects: add={action.pos_eff}, remove={action.neg_eff}")
            current_state = action.apply(current_state)
            print(f"S{i}: {current_state}")
        
        # Check goal
        print(f"\nGoal satisfied: {self.goal_satisfied(current_state)}")
        print(f"States explored: {self.states_explored}")

# Run the planner
planner = SimplePlanner(initial_state, goal, actions)
plan, final_state = planner.bfs_plan()
planner.print_plan(plan, final_state)

## Task 3: Test the Planner

In [ ]:
# TEST A: Original solvable problem
print("\n" + "="*70)
print("TEST A: Solvable Problem (Original Warehouse)")
print("="*70)

planner_a = SimplePlanner(initial_state, goal, actions)
plan_a, final_a = planner_a.bfs_plan()

if plan_a:
    print(f"\n✓ Plan found: {[str(a) for a in plan_a]}")
    print(f"  Plan length: {len(plan_a)}")
    print(f"  States explored: {planner_a.states_explored}")
    
    # Verify every action is valid
    current = initial_state.copy()
    print("\nPlan verification:")
    for i, action in enumerate(plan_a, 1):
        if action.is_applicable(current):
            print(f"  Step {i}: {action} ✓ (applicable)")
            current = action.apply(current)
        else:
            print(f"  Step {i}: {action} ✗ (NOT applicable)")
    
    if planner_a.goal_satisfied(current):
        print(f"\n✓ TEST A PASSED: Goal satisfied")
    else:
        print(f"\n✗ TEST A FAILED: Goal not satisfied")
else:
    print("\n✗ TEST A FAILED: No plan found")

In [ ]:
# TEST B: Impossible problem (no PickUp action)
print("\n" + "="*70)
print("TEST B: Impossible Problem (No PickUp Action)")
print("="*70)

# Create planner without PickUp actions
actions_no_pickup = [a for a in actions if "PickUp" not in str(a)]

planner_b = SimplePlanner(initial_state, goal, actions_no_pickup)
plan_b, final_b = planner_b.bfs_plan()

if plan_b is None:
    print(f"\n✓ TEST B PASSED: Correctly reported 'No plan found'")
    print(f"  States explored: {planner_b.states_explored}")
else:
    print(f"\n✗ TEST B FAILED: Should not find plan without PickUp")
    print(f"  Found plan: {[str(a) for a in plan_b]}")

In [ ]:
# TEST C: Irrelevant actions - check robot reaching C ≠ package at C
print("\n" + "="*70)
print("TEST C: Irrelevant Actions (Robot Position ≠ Package Position)")
print("="*70)

# Add extra action that moves robot but not package
actions_with_extra = actions + [
    Action("Move(A,C)",
           {"At(Robot,A)"},
           set(),
           {"At(Robot,C)"},
           {"At(Robot,A)"}),
]

planner_c = SimplePlanner(initial_state, goal, actions_with_extra)
plan_c, final_c = planner_c.bfs_plan()

if plan_c:
    print(f"\n✓ TEST C PASSED: Plan found despite extra action")
    print(f"  Plan: {[str(a) for a in plan_c]}")
    print(f"  Final state: {final_c}")
    
    # Verify final state has package at C
    if "At(Package,C)" in final_c:
        print(f"  Package correctly at C ✓")
    else:
        print(f"  ✗ Package NOT at C")
else:
    print(f"\n✗ TEST C FAILED: Should find plan")

## Task 4: Logic and Search in Planning

### Understanding How Logic and Search Work Together:

```
Current state
    ↓
Check action preconditions  [LOGIC]
    ↓
Generate applicable actions [LOGIC]
    ↓
Generate successor state    [LOGIC]
    ↓
Search over alternatives    [SEARCH]
    ↓
Goal?                        [SEARCH]
```

### The Role of Each Component:

**Logic determines what is POSSIBLE:**
- Check if state satisfies preconditions: $S \models \text{Preconditions}(a)$
- Compute effects: $S' = \text{Apply}(S, a)$
- Verify goal: $S \models G$?

**Search determines what to TRY:**
- Explore which actions to apply first (BFS = breadth-first, random = depth-first)
- Which states to revisit (frontier management)
- When to stop (goal test)

### Answer to "Complete the description":

```
Current state
    ↓
Check action preconditions
    ↓
Filter to applicable actions  [← THIS STEP]
    ↓
Generate successor state
    ↓
Search over alternatives
    ↓
Goal?
```

**In our planner:**
- **Logic:** The `is_applicable()` method checks preconditions and the `apply()` method computes effects
- **Search:** The BFS loop explores states, trying all applicable actions from each state

In [ ]:
print("="*70)
print("Demonstrating Logic vs Search")
print("="*70)

print("""
Example from our warehouse problem:

State S₁ after PickUp(Package, A):
  {At(Robot,A), Holding(Package)}

Which actions are applicable?

Action: Move(A,B)
  Precondition: At(Robot,A)
  In state? YES ✓
  Applicable? YES → successor is {At(Robot,B), Holding(Package)}

Action: Move(B,C)
  Precondition: At(Robot,B)
  In state? NO ✗
  Applicable? NO

Action: Drop(Package,A)
  Preconditions: At(Robot,A), Holding(Package)
  In state? At(Robot,A) YES, Holding(Package) YES ✓
  Applicable? YES → successor is {At(Robot,A), At(Package,A)}

Logic tells us: Only Move(A,B) and Drop(Package,A) are possible here.

Search then chooses: Should we do Move(A,B) or Drop(Package,A) first?
  - BFS: Try both, explore breadth-first
  - DFS: Go deep first (e.g., Move(A,B) then Move(B,C))
  - A*: Try most promising first (guided by heuristic)

Both actions are logically valid, but search picks which to explore.
""")

## Task 5: Optional - Can the LLM Verify Its Own Plan?

This is a critical test: Does the LLM correctly explain why its plan works?

In [ ]:
print("="*70)
print("LLM Self-Verification Test")
print("="*70)

print("""
Question to LLM:
  "For the plan [PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C)],
   explain why each action's preconditions are satisfied."

Correct LLM Response Should Be:
  
  Step 1: PickUp(Package,A)
    Preconditions: At(Robot,A), At(Package,A)
    State before: {At(Robot,A), At(Package,A)}
    Both preconditions satisfied? YES ✓
  
  Step 2: Move(A,B)
    Preconditions: At(Robot,A)
    State after Step 1: {At(Robot,A), Holding(Package)}
    Precondition satisfied? YES ✓
  
  Step 3: Move(B,C)
    Preconditions: At(Robot,B)
    State after Step 2: {At(Robot,B), Holding(Package)}
    Precondition satisfied? YES ✓
  
  Step 4: Drop(Package,C)
    Preconditions: At(Robot,C), Holding(Package)
    State after Step 3: {At(Robot,C), Holding(Package)}
    Both preconditions satisfied? YES ✓

Key Point: The engineer should TRUST the independent verification more than
the LLM's explanation, because:

  1. The LLM might make logical errors in explanation
  2. The independent verification (running the code) is objective
  3. An explanation can "sound right" but be semantically wrong
  4. Engineering principle: Execution beats explanation
""")

## Task 6 & 7 & 8: Optional Prolog Extension (If SWI-Prolog Available)

Prolog provides an independent logical verifier for planning.

In [ ]:
import subprocess
import os

# Create Prolog knowledge base for warehouse
prolog_code = """
% Facts: which locations are connected
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

% Rule: can move from X to Y if connected
can_move(X, Y) :-
    connected(X, Y).

% Queries to test
% ?- can_move(a, b).  % Should be true
% ?- can_move(a, c).  % Should be false
% ?- can_move(b, c).  % Should be true
"""

# Write Prolog file
prolog_file = "/home/claude/warehouse_planner.pl"
with open(prolog_file, 'w') as f:
    f.write(prolog_code)

print(f"Created Prolog knowledge base: {prolog_file}")
print("\nProlog Code:")
print(prolog_code)

In [ ]:
# Test Prolog queries
print("="*70)
print("Prolog Verification: Testing can_move Rules")
print("="*70)

queries = [
    ("can_move(a, b)", True),   # Should succeed
    ("can_move(a, c)", False),  # Should fail (no direct connection)
    ("can_move(b, c)", True),   # Should succeed
    ("can_move(c, a)", False),  # Should fail
]

print("\nProlog Query Results:")

for query, expected in queries:
    try:
        # Run SWI-Prolog with query
        result = subprocess.run(
            ["swipl", "-g", f"({query} -> write('true'); write('false')), halt", 
             "-t", "halt(1)", prolog_file],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        output = result.stdout.strip()
        success = output == "true"
        
        status = "✓" if success == expected else "✗"
        print(f"{status} ?- {query:20} => {output:5} (expected: {str(expected).lower()})")
    except Exception as e:
        print(f"✗ Query {query} failed: {e}")

print("\n" + "="*70)
print("Prolog Analysis")
print("="*70)
print("""
Why does ?- can_move(a, c) fail?
  - Rule: can_move(X, Y) :- connected(X, Y)
  - Query: can_move(a, c)
  - Prolog checks: Is there a fact connected(a, c)?
  - Answer: NO (only connected(a,b) and connected(b,c) exist)
  - Therefore: can_move(a, c) fails

Key Point: Prolog provides INDEPENDENT verification that the Python
planner's moves are logically sound. If the Python planner suggests
Move(a, c), Prolog would reject it as invalid.
""")

In [ ]:
# Extend Prolog with more complex rules
prolog_extended = """
% Facts: warehouse connections
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

% Direct move rule
can_move(X, Y) :- connected(X, Y).

% Multi-step path (using recursion)
path(X, X).  % Base case: path to self
path(X, Y) :-
    can_move(X, Z),
    path(Z, Y).

% Valid move for a plan
valid_move(X, Y) :- connected(X, Y).

% Check a sequence of moves
valid_sequence([]).  % Empty sequence is valid
valid_sequence([X-Y|Rest]) :-
    valid_move(X, Y),
    valid_sequence(Rest).
"""

# Write extended Prolog
prolog_extended_file = "/home/claude/warehouse_planner_extended.pl"
with open(prolog_extended_file, 'w') as f:
    f.write(prolog_extended)

print("\n" + "="*70)
print("Extended Prolog: Path Finding and Plan Validation")
print("="*70)

print("\nExtended Prolog Rules:")
print(prolog_extended)

# Test path queries
path_queries = [
    ("path(a, b)", True),   # Direct connection
    ("path(a, c)", True),   # Via b
    ("path(c, a)", True),   # Via b (symmetric)
]

print("\nPath Finding Queries:")
for query, expected in path_queries:
    try:
        result = subprocess.run(
            ["swipl", "-g", f"({query} -> write('true'); write('false')), halt",
             "-t", "halt(1)", prolog_extended_file],
            capture_output=True,
            text=True,
            timeout=5
        )
        output = result.stdout.strip()
        success = output == "true"
        status = "✓" if success == expected else "✗"
        print(f"{status} ?- {query:20} => {output}")
    except Exception as e:
        print(f"✗ {query}: {e}")

## Task 8: Prolog Logical Reasoning Example

In [ ]:
print("="*70)
print("Prolog Logical Reasoning: Facts and Rules")
print("="*70)

prolog_logic_example = """
Facts and Rules:
  wet_road.                    % Fact: The road is wet
  slippery :- wet_road.        % Rule: If wet_road then slippery
  reduce_speed :- slippery.    % Rule: If slippery then reduce_speed

Query: ?- reduce_speed.

Prolog's Reasoning Chain:
  1. To prove reduce_speed, check the rule:
     reduce_speed :- slippery.
     So we need to prove: slippery?
  
  2. To prove slippery, check the rule:
     slippery :- wet_road.
     So we need to prove: wet_road?
  
  3. Check the facts:
     wet_road.  ← Found! This is a fact.
  
  4. Backtrack with success:
     wet_road is true
     → slippery is true (by rule)
     → reduce_speed is true (by rule)
  
  5. Query succeeds!

Logical Chain:
  wet_road ⇒ slippery ⇒ reduce_speed
  (true)   ⇒ (true)    ⇒ (true)

This demonstrates Facts + Rules → Inference → Query Answer
"""

print(prolog_logic_example)

# Create and test this example in Prolog
logic_example_pl = """
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
"""

logic_file = "/home/claude/logic_example.pl"
with open(logic_file, 'w') as f:
    f.write(logic_example_pl)

print("\nTesting in SWI-Prolog:")
try:
    result = subprocess.run(
        ["swipl", "-g", "(reduce_speed -> write('SUCCESS: reduce_speed is true'); write('FAIL')), halt",
         "-t", "halt(1)", logic_file],
        capture_output=True,
        text=True,
        timeout=5
    )
    print(result.stdout.strip())
except Exception as e:
    print(f"Error: {e}")

## Task 9: Reflection on LLM-Assisted Planning

### Reflection Questions and Answers:

**1. Why is it useful to specify action preconditions and effects before asking an LLM?**

Because:
- It forces you to think clearly about what each action does
- The LLM can then correctly implement the semantics
- Without clear specs, the LLM might generate nonsensical code
- Preconditions/effects are the "contract" the algorithm depends on

**2. Give an example of an error if preconditions aren't checked:**

Without checking preconditions:
- Could execute Drop(Package, C) when Holding(Package) is false
- Could execute Move(B, C) when At(Robot, B) is false
- Would allow invalid state transitions
- Planner might "find" plans that don't actually work

**3. Why is a plan that "looks reasonable" not necessarily valid?**

Because:
- The sequence might violate preconditions
- A human might skip steps when describing a plan
- "Looks right" is not the same as "mathematically verified"
- Only execution/verification proves validity

**4. What did the LLM contribute?**

Typically:
- Boilerplate code structure
- BFS algorithm loop
- Action application logic
- State representation ideas

**5. What did you have to verify independently?**

Everything:
- Does it find a solution when one exists?
- Does it report "no solution" when impossible?
- Are preconditions actually checked?
- Does the plan actually achieve the goal?
- Are there bugs in effect application?

**6. Where is logical reasoning being used?**

In:
- Checking if state satisfies preconditions: S ⊨ Preconditions(a)
- Computing effects
- Testing goal satisfaction: S ⊨ G

**7. How is planning related to search algorithms?**

Planning IS search:
- States are problem configurations
- Actions are transitions
- Goal is the target state
- BFS finds shortest plan
- Could use A* with distance-to-goal heuristic

## Summary

### What We Learned:

✓ **Planning as a Formal Problem**: (I, A, G) with preconditions and effects  
✓ **Logic in Planning**: Checking applicability and computing effects  
✓ **Search in Planning**: BFS explores sequences of applicable actions  
✓ **Implementation**: Reference implementation matches specification exactly  
✓ **Testing**: Multiple tests catch edge cases and bugs  
✓ **Verification**: Prolog provides independent logical verification  
✓ **LLM Role**: Useful for coding, but engineer must understand and validate  

### Key Principle:

```
Logic + Search = Planning
  ↓        ↓
What's   Which      Sequence of
possible to try  →  actions that
in this  first      achieves goal
state
```

### Engineering Lesson:

**Understand → Specify → Generate → Execute → Verify**

The LLM is a tool that accelerates Step 3 (Generate), but all other steps remain your responsibility as an engineer.